In [2]:
import pandas as pd
import threading, json
from openai import OpenAI #estamos la clase concreta OpenAI del módulo openai
from dotenv import load_dotenv #importamos una función concreta del módulo
load_dotenv("template.env")

# Acceder a la clave de API de OpenAI
api_key = os.getenv("OPENAI_API_KEY")

# Asegurarte de que la clave de API se haya cargado correctamente
if api_key is None:
    raise ValueError("La clave de API no está configurada en el archivo .env")
    
client = OpenAI() #creando un objeto de la clase

#Importar fine-tuned job
f_t_job = client.fine_tuning.jobs.retrieve("ftjob-rHxNZYkt5qaQ3fnxKCRnwIOO")
fine_tuned_model_id = f_t_job.fine_tuned_model

In [3]:
#PROMPTS

#AGE PROMPT
categorize_system_prompt_paraphrase ='''
La edad de adquisición (AoA) de una palabra se refiere a la edad en la que se aprendió una palabra por primera vez. 
En concreto, cuándo una persona habría entendido por primera vez esa palabra si alguien la hubiera utilizado delante de ella, incluso cuando aún no la hubiera dicho, leído o escrito. 
Calcule la edad media de adquisición (AoA) de la palabra {palabra} para un hablante nativo de español.

El formato de salida debe ser un objeto JSON: {AoA: número //AoA de la palabra expresado en años, puede incluir decimales, Word: palabra //string}
'''

In [4]:
dataset_folder = os.getenv("DATASET_FOLDER")
dataset_path = str(dataset_folder) + "FinalResults-AoA-no-FT-prompt-JSON.xlsx"

df = pd.read_excel(dataset_path)
df = df.sample(100)
df.head()

,Word,AoA,Source
8693,angustiosa,8.5,Stadthagen
89967,ostentosamente,10.5,Dictionary_on_web
79832,marimbista,10.5,RAE_dic
17392,benjamita,5.0,Dictionary_on_web
120834,traspilastra,12.5,RAE_dic


In [ ]:
#FUNCTIONS

# Json object to message array
def create_message_arr(json_object):
	word = json_object["Word"]
	m_obj = json.dumps({"palabra":word})
	test_messages = [] 
	test_messages.append({"role": "system", "content": categorize_system_prompt_paraphrase})
	test_messages.append({"role": "user", "content": m_obj})
	return test_messages

def next_line():
    global counter
    global obtain_lock
    global completed_task
    obtain_lock.acquire()
    try:
        counter += 1
        while(counter in completed_task):
            counter += 1
    finally:
        obtain_lock.release()
    return counter

def add_line(json_object,file_name):
    global results
    global res_lock
    res_lock.acquire()
    try:
        results.append(json_object)
        with open(file_name, "a") as myfile:
            myfile.write(json.dumps(json_object))
            myfile.write("\n")
    finally:
        res_lock.release()

class RequestThread(threading.Thread):
    def __init__(self,id,out_file):
        super().__init__()
        self.id = id
        self.alive = True
        self.out_file = out_file

    def kill(self):
        self.alive = False
        
    def run(self):
        while(self.alive):
            try:
                counter = next_line()
                json_object = df.iloc[counter]
                test_messages = create_message_arr(json_object)
                response = client.chat.completions.create(
                    model = fine_tuned_model_id, messages = test_messages, temperature = 0
                )
                AoA = response.choices[0].message.content
                Word = json.loads(test_messages[1]["content"])["palabra"]
                json_object = {
                    "Word":Word,
                    "NotFtAoA":json_object["AoA"],
                    "AoA":AoA,
                    "Source":json_object["Source"],
				}
                add_line(json_object,self.out_file)
                save_counter_update(counter)
            except:
                print("ERROR: kill thread (" +str(self.id) + ") due to warning")
                self.kill()

def save_counter_update(line):
    global completed_task
    global save_counter
    global save_lock
    save_lock.acquire()
    try:
        if (line == save_counter):
            save_counter+=1
            while(save_counter in completed_task):
                completed_task.remove(save_counter)
                save_counter+=1
        else:
            completed_task.append(line)
    finally:
        save_lock.release()

In [ ]:
# INITIALIZE FAIL-SAFE VARIABLES (resets progress)
save_counter = 0 # Tasks finished in order without "gaps"
completed_task=[] # Tasks completed out of order
save_lock = threading.Lock()

In [ ]:
# DISPLAY FAIL-SAFE VARIABLES
print(save_counter)
print(completed_task)

0
[]


In [ ]:
# RUN THREADS
# Define provisional file
file_prov_name = "middle_files/prov_output_counter.jsonl"

# Define line counter and lock
counter = save_counter-1
obtain_lock = threading.Lock()

# Define output array and lock 
results = []
res_lock = threading.Lock()

# Define number of lines in file
max_lines = len(df)

# Define number of threads
num_Threads = 3

# Create and start threads
threads = []
for i in range(0,num_Threads):
    thread = RequestThread(i,file_prov_name)  
    threads.append(thread)
for i in range(0,num_Threads):
    threads[i].start()
   
# Wait until all lines are done working
while (counter<max_lines):
    pass

# Kill and join Threads
for i in range(0,num_Threads):
    threads[i].kill()
for i in range(0,num_Threads):
    threads[i].join()
del threads
print("success")

kill thread due to warning: 2
success


In [ ]:
# KILL REMAINING THREADS (if process finishes abruptly)
for i in range(0,num_Threads):
    threads[i].kill()
for i in range(0,num_Threads):
    threads[i].join()
print("kill process")

kill process


In [ ]:
# PROVISIONAL OUTPUT FILE TO EXCEL
f = open(file_prov_name, "r")
res = []
for line in f:
	json_object = json.loads(line)
	res.append(json_object)
clean_dtset = pd.DataFrame(res)

file_name = "output_files/ProvResults_all_words.xlsx"

with pd.ExcelWriter(file_name) as writer:
	clean_dtset.to_excel(writer, sheet_name='Results',index=False)
print("")